## 과제 2. 지난 5년간 서울시 전세 가격 변화 분석

### 분석 자료
완결된 최근 5개 연도인 **2021~2025년**의 서울 아파트 전세가격지수를 비교한다. 단위는 2021년 6월=100이다.

출처: [국가지표체계 — 주택유형별 주택 전세가격 지수](https://www.index.go.kr/unity/potal/eNara/sub/showStblGams3.do?freq=Y&idx_cd=F0183&period=N&stts_cd=F018301) (한국부동산원)

이 자료는 서울 아파트의 가격 변화 지표이며, 개별 계약의 평균 전세 보증금과는 구분된다.


In [56]:
# 이 셀부터 순서대로 실행하면 됩니다.
import pandas as pd
import plotly.express as px

# 출처 표의 서울 아파트 전세가격지수 연간 값
jeonse = pd.DataFrame({
    "연도": [2021, 2022, 2023, 2024, 2025],
    "전세가격지수": [109.5, 112.9, 91.4, 96.8, 100.6]
})
jeonse["전년 대비 변화율(%)"] = (
    jeonse["전세가격지수"].pct_change() * 100
).round(1)
jeonse


,연도,전세가격지수,전년 대비 변화율(%)
0,2021,109.5,NaN
1,2022,112.9,3.1
2,2023,91.4,-19.0
3,2024,96.8,5.9
4,2025,100.6,3.9


### 연도별 지수와 변화율 표
전년 대비 변화율은 (당해 연도 지수 ÷ 전년도 지수 − 1) × 100으로 계산한다. 2021년은 비교할 전년 값을 입력하지 않았으므로 결측값으로 표시한다.


In [57]:
jeonse


,연도,전세가격지수,전년 대비 변화율(%)
0,2021,109.5,NaN
1,2022,112.9,3.1
2,2023,91.4,-19.0
3,2024,96.8,5.9
4,2025,100.6,3.9


### 선 그래프: 전세가격지수 추이
각 점에 마우스를 올리면 해당 연도의 지수를 확인할 수 있다.


In [58]:
import plotly.graph_objects as go

# 연도별 색상: 고점은 보라색, 저점은 코랄색, 회복 구간은 청록색과 파란색
point_colors = ["#818CF8", "#8B5CF6", "#F97373", "#14B8A6", "#2563EB"]

fig = go.Figure(go.Scatter(
    x=jeonse["연도"], y=jeonse["전세가격지수"],
    mode="lines+markers+text",
    line=dict(color="#6366F1", width=4),
    marker=dict(size=15, color=point_colors,
                line=dict(color="white", width=3)),
    text=jeonse["전세가격지수"],
    texttemplate="<b>%{text:.1f}</b>",
    textposition="top center",
    textfont=dict(size=15, color="#334155"),
    cliponaxis=False,
    hovertemplate="<b>%{x}년</b><br>전세가격지수 %{y:.1f}<extra></extra>"
))

# 회복 구간을 은은하게 강조
fig.add_vrect(x0=2023, x1=2025.25, fillcolor="#14B8A6",
              opacity=0.06, line_width=0, layer="below")
fig.add_annotation(
    x=2024, y=119, text="<b>2024~2025 · 2년 연속 반등</b>",
    showarrow=False, font=dict(size=13, color="#0F766E"),
    bgcolor="#E6F7F3", borderpad=8
)
fig.update_layout(
    title=dict(
        text="<b>서울 전세, 하락 이후 회복 중</b>"
             "<br><sup>서울 아파트 전세가격지수 · 2021~2025 · 2021.6=100</sup>",
        x=0.04, y=0.95, font=dict(size=23, color="#172554")
    ),
    template="plotly_white", height=510,
    paper_bgcolor="#F5F7FC", plot_bgcolor="white",
    font=dict(family="Malgun Gothic, sans-serif", size=13, color="#475569"),
    margin=dict(l=65, r=40, t=115, b=85),
    hoverlabel=dict(bgcolor="#172554", font_color="white", font_size=14),
    showlegend=False
)
fig.update_xaxes(
    title=None, tickvals=jeonse["연도"], ticksuffix="년",
    range=[2020.7, 2025.3], showgrid=False,
    zeroline=False, tickfont=dict(size=13)
)
fig.update_yaxes(
    title="전세가격지수", range=[80, 125], dtick=10,
    gridcolor="#E8EDF5", griddash="dot", zeroline=False
)
fig.add_annotation(
    x=0, y=-0.20, xref="paper", yref="paper", xanchor="left",
    text="자료: 한국부동산원 / 국가지표체계 · 변화를 보기 위해 세로축은 80부터 표시",
    showarrow=False, font=dict(size=11, color="#64748B")
)
fig.show()


### 막대 그래프: 전년 대비 변화율
상승은 초록색, 하락은 빨간색으로 표시한다.


In [59]:
change = jeonse.dropna(subset=["전년 대비 변화율(%)"]).copy()
change["방향"] = change["전년 대비 변화율(%)"].apply(
    lambda value: "상승" if value >= 0 else "하락"
)

# 상승·하락에 따라 막대 색상을 다르게 설정
bar_colors = [
    "#14B8A6" if value >= 0 else "#F97373"
    for value in change["전년 대비 변화율(%)"]
]
fig = go.Figure(go.Bar(
    x=change["연도"], y=change["전년 대비 변화율(%)"],
    customdata=change["방향"],
    marker=dict(color=bar_colors, cornerradius=8),
    width=0.52,
    text=change["전년 대비 변화율(%)"],
    texttemplate="<b>%{text:+.1f}%</b>", textposition="outside",
    textfont=dict(size=17, color="#334155"), cliponaxis=False,
    hovertemplate="<b>%{x}년 · %{customdata}</b>"
                  "<br>전년 대비 %{y:+.1f}%<extra></extra>"
))
fig.add_hline(y=0, line_width=1.5, line_color="#94A3B8")
fig.update_layout(
    title=dict(
        text="<b>2023년 급락, 이후 2년 연속 반등</b>"
             "<br><sup>서울 아파트 전세가격지수의 전년 대비 변화율 · 초록: 상승 / 코랄: 하락</sup>",
        x=0.04, y=0.95, font=dict(size=23, color="#172554")
    ),
    template="plotly_white", height=510,
    paper_bgcolor="#F5F7FC", plot_bgcolor="white",
    font=dict(family="Malgun Gothic, sans-serif", size=13, color="#475569"),
    margin=dict(l=65, r=40, t=115, b=85),
    hoverlabel=dict(bgcolor="#172554", font_color="white", font_size=14),
    showlegend=False
)
fig.update_xaxes(
    title=None, tickvals=change["연도"], ticksuffix="년",
    showgrid=False, zeroline=False, tickfont=dict(size=13)
)
fig.update_yaxes(
    title="전년 대비 변화율", ticksuffix="%", range=[-25, 12],
    dtick=5, gridcolor="#E8EDF5", griddash="dot", zeroline=False
)
fig.add_annotation(
    x=0, y=-0.20, xref="paper", yref="paper", xanchor="left",
    text="자료: 한국부동산원 / 국가지표체계 · 변화율 = (당해 지수 ÷ 전년 지수 − 1) × 100",
    showarrow=False, font=dict(size=11, color="#64748B")
)
fig.show()


### 분석 결과
2021년 109.5에서 2022년 112.9로 상승한 뒤, 2023년 91.4로 전년 대비 19.0% 하락했다. 이후 2024년에는 5.9%, 2025년에는 3.9% 반등했다.

2025년 지수는 2022년 고점보다 약 10.9% 낮다. 따라서 이 기간은 **상승 → 하락 → 부분 회복**의 흐름으로 설명할 수 있다.

분석 범위는 서울 아파트이며, 단독·연립·다세대 주택을 포함한 서울 전체 주택으로 일반화할 때는 주의가 필요하다. 연간 자료만으로 월별 전환 시점이나 가격 변화의 원인을 확정할 수는 없다.
